# 판정 불명 Withdrawn 민감도 분석 (`06_Sensitivity_Ambiguous`)

```
05_Intervention_Guide → [06_Sensitivity_Ambiguous]
```

## 목적 (열린 이슈 해소)

최종 `Withdrawn`이지만 `date_unregistration`(취소일)이 없는 **93명**은 25일에 재학 중이었는지 알 수 없어 주 코호트에서 제외했다.
그런데 `03_Landmark25_EDA` 13절에서 **93명 중 87명이 holdout 학기(2014J)**에 몰려 있음을 확인했다. 이 노트북은 두 가지를 확인한다.

1. **이 학생들은 어떤 학생인가**: 25일까지의 행동이 "25일 전에 떠난 학생"에 가까운가, "25일 이후 이탈자"에 가까운가
2. **holdout 결과가 얼마나 흔들리는가**: 87명을 "25일 재학 후 이탈"로 포함했을 때 성능·등급·명단 포착률이 어떻게 바뀌는가

## 방법

- 제외 세그먼트는 통합 정본에 피처가 비어 있으므로, **`02_Integrated_Table`과 같은 정의로 25일 피처를 다시 계산하는 함수**를 만든다.
  이 함수로 **기존 모델 대상 27,661건의 피처를 재계산해 통합 정본과 완전히 일치하는지** 먼저 확인한 뒤에만 제외 세그먼트에 적용한다.
- 최종 모델(XGBoost + sigmoid 보정)과 확정된 1·2차 규칙을 **그대로** 적용한다. 모델·규칙은 바꾸지 않는다.

## 1. 환경 설정과 원본 로드

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
TIER_NAMES = {1: '1 고위험', 2: '2 주의', 3: '3 관찰', 4: '4 일반'}


def show(frame, formats=None):
    view = frame.copy()
    for col, f in (formats or {}).items():
        if col in view.columns:
            view[col] = view[col].map(lambda v, f=f: f.format(v) if pd.notna(v) else '')
    display(view)


PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'CSV_files').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '../..').resolve()
CSV_DIR = PROJECT_ROOT / 'CSV_files'
DATA_DIR, MODEL_DIR = CSV_DIR / '통합 버전', CSV_DIR / '모델링'
KEYS = ['code_module', 'code_presentation', 'id_student']
LANDMARK_N = 25

spec = json.load(open(DATA_DIR / 'feature_spec_n25.json', encoding='utf-8'))
rule = json.load(open(MODEL_DIR / 'final_tier_rule_n25.json', encoding='utf-8'))
cp_rule = json.load(open(MODEL_DIR / 'second_checkpoint_rule_n25.json', encoding='utf-8'))
all_cohorts = pd.read_csv(DATA_DIR / 'landmark25_all_cohorts.csv')
clean = pd.read_csv(DATA_DIR / 'model_df_clean_n25.csv')
oof = pd.read_csv(MODEL_DIR / 'dev_oof_predictions_n25.csv')
hold_tiers = pd.read_csv(MODEL_DIR / 'final_holdout_tiers_n25.csv')
cp_students = pd.read_csv(MODEL_DIR / 'second_checkpoint_students_n25.csv')
assessments = pd.read_csv(CSV_DIR / 'assessments.csv')
student_assessment = pd.read_csv(CSV_DIR / 'studentAssessment.csv')
parts = []
for chunk in pd.read_csv(CSV_DIR / 'studentVle.csv', chunksize=2_000_000):
    parts.append(chunk[(chunk['date'] >= 0) & (chunk['date'] <= LANDMARK_N)])
vle_window = pd.concat(parts, ignore_index=True)

TARGET, NUM, CAT = spec['target'], spec['numeric_features'], spec['categorical_features']
FEATURES = NUM + CAT
seg = all_cohorts['cohort_status_25'].value_counts()
print(seg)
amb = all_cohorts[all_cohorts['cohort_status_25'] == 'excluded_ambiguous_withdrawn']
print('판정 불명 학기 분포:', amb['code_presentation'].value_counts().to_dict())

cohort_status_25
model_eligible                  27661
excluded_early_leaver            4819
excluded_ambiguous_withdrawn       93
excluded_late_registration         20
Name: count, dtype: int64
판정 불명 학기 분포: {'2014J': 87, '2014B': 3, '2013J': 2, '2013B': 1}


## 2. 25일 피처 재계산 함수와 정본 일치 검증

`02_Integrated_Table` 4·5절 정의(0913_01)를 그대로 함수로 옮긴다. 모델 대상 27,661건에 적용해 **모든 피처 컬럼이 통합 정본과 같아야** 제외 세그먼트에 쓸 수 있다.

In [2]:
FEATURE_COLS = ['total_click_25', 'active_days_25', 'distinct_resources_25', 'no_vle_activity_25', 'n_opportunity_25', 'n_submitted_25',
                'n_missing_25', 'submission_rate_25', 'avg_score_25', 'avg_submit_delay_25', 'no_assessment_opportunity_25', 'n_banked_25']


def build_features_25(rows):
    out = rows[KEYS].copy()
    vle = (vle_window.groupby(KEYS).agg(total_click_25=('sum_click', 'sum'), active_days_25=('date', 'nunique'), distinct_resources_25=('id_site', 'nunique')).reset_index())
    out = out.merge(vle, on=KEYS, how='left')
    for c in ['total_click_25', 'active_days_25', 'distinct_resources_25']:
        out[c] = out[c].fillna(0)
    out['no_vle_activity_25'] = (out['total_click_25'] == 0).astype(int)

    opp = assessments[assessments['assessment_type'].isin(['TMA', 'CMA']) & (assessments['date'] <= LANDMARK_N)]
    mod_n = opp.groupby(['code_module', 'code_presentation']).size().rename('n_module_opportunity_25').reset_index()
    meta = opp[['id_assessment', 'code_module', 'code_presentation', 'date']].rename(columns={'date': 'due_date_25'})
    banked = student_assessment[(student_assessment['is_banked'] == 1) & student_assessment['id_assessment'].isin(opp['id_assessment'])].merge(meta, on='id_assessment')
    banked = banked.groupby(KEYS).size().rename('n_banked_25').reset_index()
    out = out.merge(mod_n, on=['code_module', 'code_presentation'], how='left').merge(banked, on=KEYS, how='left')
    out['n_module_opportunity_25'] = out['n_module_opportunity_25'].fillna(0).astype(int)
    out['n_banked_25'] = out['n_banked_25'].fillna(0).astype(int)
    out['n_opportunity_25'] = out['n_module_opportunity_25'] - out['n_banked_25']
    out['no_assessment_opportunity_25'] = (out['n_opportunity_25'] == 0).astype(int)

    sub = student_assessment[(student_assessment['is_banked'] == 0) & (student_assessment['date_submitted'] <= LANDMARK_N) & student_assessment['id_assessment'].isin(opp['id_assessment'])].merge(meta, on='id_assessment')
    sub['submit_delay_25'] = sub['date_submitted'] - sub['due_date_25']
    sub = sub.groupby(KEYS).agg(n_submitted_25=('id_assessment', 'count'), avg_score_25=('score', 'mean'), avg_submit_delay_25=('submit_delay_25', 'mean')).reset_index()
    out = out.merge(sub, on=KEYS, how='left')
    out['n_submitted_25'] = out['n_submitted_25'].fillna(0).astype(int)
    out['n_missing_25'] = out['n_opportunity_25'] - out['n_submitted_25']
    out['submission_rate_25'] = np.where(out['n_opportunity_25'] > 0, out['n_submitted_25'] / out['n_opportunity_25'], np.nan)
    return out.drop(columns='n_module_opportunity_25')


eligible = all_cohorts[all_cohorts['cohort_status_25'] == 'model_eligible']
rebuilt = build_features_25(eligible)
cmp = eligible[KEYS + FEATURE_COLS].merge(rebuilt, on=KEYS, suffixes=('_정본', '_재계산'), validate='one_to_one')
mismatch = {c: int((~np.isclose(cmp[c + '_정본'].astype(float), cmp[c + '_재계산'].astype(float), equal_nan=True)).sum()) for c in FEATURE_COLS}
print('정본 vs 재계산 불일치 행 수:', mismatch)
assert len(cmp) == len(eligible) == 27_661 and sum(mismatch.values()) == 0, '재계산 함수가 02 정의와 다름'
print('검증 통과 — 재계산 함수가 통합 정본 피처와 완전히 일치')

정본 vs 재계산 불일치 행 수: {'total_click_25': 0, 'active_days_25': 0, 'distinct_resources_25': 0, 'no_vle_activity_25': 0, 'n_opportunity_25': 0, 'n_submitted_25': 0, 'n_missing_25': 0, 'submission_rate_25': 0, 'avg_score_25': 0, 'avg_submit_delay_25': 0, 'no_assessment_opportunity_25': 0, 'n_banked_25': 0}
검증 통과 — 재계산 함수가 통합 정본 피처와 완전히 일치


## 3. 판정 불명 학생은 어떤 학생인가

같은 함수로 **판정 불명 93명**과 **25일까지 이탈해 제외된 학생(4,819명)**의 25일 피처를 계산하고, 모델 대상의 25일 이후 이탈자·비이탈자와 비교한다.

- 판정 불명 학생의 행동이 **25일 전 이탈자와 비슷하면**(활동·제출이 거의 없음) 25일에 이미 떠났을 가능성이 높아 **제외가 타당**하다.
- **25일 이후 이탈자와 비슷하면** holdout 이탈자가 과소 집계됐을 가능성이 크다.

주의: 25일 전 이탈자는 25일 이전에 취소했으므로 관측 기간이 짧아 활동이 적은 것이 당연하다. 비교는 경향 확인용이다.

In [3]:
early = all_cohorts[all_cohorts['cohort_status_25'] == 'excluded_early_leaver']
feat_amb = amb[KEYS + ['final_result', 'date_registration', 'studied_credits', 'num_of_prev_attempts']].merge(build_features_25(amb), on=KEYS)
feat_early = early[KEYS + ['date_registration', 'date_unregistration']].merge(build_features_25(early), on=KEYS)
elig_feat = clean.copy()


def profile(frame, name):
    has_opp = frame['n_opportunity_25'] > 0
    return {'집단': name, '학생': len(frame), '2014J 비율': (frame['code_presentation'] == '2014J').mean(),
            '25일 무활동': (frame['total_click_25'] == 0).mean(), '클릭 중앙값': frame['total_click_25'].median(),
            '활동일 중앙값': frame['active_days_25'].median(),
            '기회 있는데 미제출': (has_opp & (frame['n_submitted_25'] == 0)).sum() / max(has_opp.sum(), 1),
            '제출자 평균 점수': frame['avg_score_25'].mean()}


prof = pd.DataFrame([
    profile(feat_amb, '판정 불명 Withdrawn (93)'),
    profile(feat_amb[feat_amb['code_presentation'] == '2014J'], '└ 그중 2014J (87)'),
    profile(feat_early, '25일까지 이탈 (제외)'),
    profile(elig_feat[elig_feat[TARGET] == 1], '모델 대상 · 25일 이후 이탈'),
    profile(elig_feat[elig_feat[TARGET] == 0], '모델 대상 · 비이탈'),
]).set_index('집단')
show(prof, {'학생': '{:,}', '2014J 비율': '{:.0%}', '25일 무활동': '{:.1%}', '클릭 중앙값': '{:.0f}', '활동일 중앙값': '{:.0f}',
            '기회 있는데 미제출': '{:.1%}', '제출자 평균 점수': '{:.1f}'})

early_2014j = feat_early[feat_early['code_presentation'] == '2014J']
print('25일까지 이탈자의 취소일 분포(2014J, 참고):', early_2014j['date_unregistration'].describe().round(1).to_dict())
print('판정 불명 2014J 과목 분포:', feat_amb.loc[feat_amb['code_presentation'] == '2014J', 'code_module'].value_counts().to_dict())

,학생,2014J 비율,25일 무활동,클릭 중앙값,활동일 중앙값,기회 있는데 미제출,제출자 평균 점수
집단,,,,,,,
판정 불명 Withdrawn (93),93,94%,40.9%,20,1,78.8%,57.9
└ 그중 2014J (87),87,100%,41.4%,20,1,80.0%,55.1
25일까지 이탈 (제외),"4,819",42%,70.3%,0,0,92.2%,58.6
모델 대상 · 25일 이후 이탈,"5,246",33%,6.5%,134,8,26.2%,64.9
모델 대상 · 비이탈,"22,408",33%,4.3%,176,9,9.6%,74.4


25일까지 이탈자의 취소일 분포(2014J, 참고): {'count': 2017.0, 'mean': -8.0, 'std': 31.8, 'min': -163.0, '25%': -16.0, '50%': 5.0, '75%': 12.0, 'max': 25.0}
판정 불명 2014J 과목 분포: {'CCC': 28, 'FFF': 24, 'DDD': 16, 'BBB': 13, 'EEE': 4, 'GGG': 2}


**해석 — 판정 불명 학생은 25일 전 이탈자에 가깝다**

| 집단 | 25일 무활동 | 활동일 중앙값 | 기회 있는데 미제출 | 제출자 평균 점수 |
|---|---:|---:|---:|---:|
| 판정 불명 (93) | 40.9% | **1일** | **78.8%** | 57.9 |
| 25일까지 이탈 (4,819) | 70.3% | 0일 | 92.2% | 58.6 |
| 25일 이후 이탈 (5,246) | 6.5% | 8일 | 26.2% | 64.9 |
| 비이탈 (22,408) | 4.3% | 9일 | 9.6% | 74.4 |

- 판정 불명 학생의 활동일 중앙값은 **1일**, 평가 미제출은 **78.8%**다. 25일 이후 이탈자(8일, 26.2%)보다 **25일 전 이탈자(0일, 92.2%)와 훨씬 비슷하다.**
  무활동 비율(40.9%)은 두 집단 사이지만, 활동이 있어도 극히 짧다.
- 따라서 이 학생들은 **25일 전후에 이미 학습을 중단했을 가능성이 높고**, 25일 재학생으로 보기 어렵다. **주 분석에서 제외한 결정은 타당**하다.
- 2014J에 몰린 이유는 원본 데이터의 마지막 학기라 **취소일 기록이 늦게 반영되지 않은 행정 지연**으로 보인다(과목 분포 CCC 28, FFF 24, DDD 16, BBB 13, EEE 4, GGG 2로 특정 과목에 몰리지 않는다). 원인은 데이터에서 확인할 수 없는 추정이다.

## 4. holdout 민감도 — 87명을 "25일 재학 후 이탈"로 포함하면

**시나리오**

| 시나리오 | 가정 | holdout 구성 |
|---|---|---|
| S0 (주 분석) | 25일 재학 여부 불명 → 제외 | 9,150건, 이탈 1,720건 |
| S1 (최대 영향) | 87명 모두 25일에 재학 중이었고 이후 이탈 | 9,237건, 이탈 1,807건 |

S1은 "과소 집계가 최대인 경우"의 상한 시나리오다. 실제는 S0과 S1 사이에 있다.
최종 모델·보정기·1차 등급 기준·2차 규칙은 모두 그대로 적용한다. 과목·학기 내 순위(관찰 조건)는 87명을 포함해 다시 계산한다.

In [4]:
df = clean.copy()
df['presentation_period'] = df['code_presentation'].str[-1]
df[TARGET] = df[TARGET].astype(int)
dev = df[df['code_presentation'] != '2014J'].reset_index(drop=True)
hold = df[df['code_presentation'] == '2014J'].reset_index(drop=True)
y_dev = dev[TARGET].to_numpy()


def _logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)


prep = ColumnTransformer([('num', 'passthrough', NUM), ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT)], sparse_threshold=0)
model = XGBClassifier(n_estimators=400, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=(y_dev == 0).sum() / (y_dev == 1).sum(),
                      tree_method='hist', eval_metric='aucpr', random_state=42, n_jobs=4)
final_model = Pipeline([('prep', prep), ('model', model)]).set_params(**rule['params']).fit(dev[FEATURES], y_dev)
calibrator = LogisticRegression(C=1e6, max_iter=1000).fit(_logit(oof['oof_XGBoost']), y_dev)
p_hold = calibrator.predict_proba(_logit(final_model.predict_proba(hold[FEATURES])[:, 1]))[:, 1]
assert np.allclose(p_hold, hold_tiers['pred_proba'].to_numpy(), atol=1e-6)

amb_j = feat_amb[feat_amb['code_presentation'] == '2014J'].copy()
amb_j = amb_j.merge(amb[KEYS + ['gender', 'region', 'highest_education', 'imd_band', 'age_band', 'disability']], on=KEYS)
amb_j['imd_band'] = amb_j['imd_band'].fillna('Unknown')
amb_j['presentation_period'] = 'J'
amb_j[TARGET] = 1
p_amb = calibrator.predict_proba(_logit(final_model.predict_proba(amb_j[FEATURES])[:, 1]))[:, 1]
print(f'판정 불명 87명 보정 확률: 평균 {p_amb.mean():.1%}, 중앙값 {np.median(p_amb):.1%} (holdout 이탈자 평균 {p_hold[hold[TARGET] == 1].mean():.1%}, 비이탈자 {p_hold[hold[TARGET] == 0].mean():.1%})')


def tiers(frame, p):
    tr = rule['tier_rule']
    pct = pd.Series(p, index=frame.index).groupby([frame['code_module'], frame['code_presentation']]).rank(pct=True, ascending=False, method='first').to_numpy()
    t = np.full(len(p), 4)
    t[(p >= tr['low']) | (pct <= tr['module_top'])] = 3
    t[p >= tr['mid']] = 2
    t[p >= tr['high']] = 1
    return t


s0 = hold[['code_module', 'code_presentation', 'id_student', TARGET]].assign(proba=p_hold, 판정불명=False)
s1 = pd.concat([s0, amb_j[['code_module', 'code_presentation', 'id_student', TARGET]].assign(proba=p_amb, 판정불명=True)], ignore_index=True)
s0['tier'] = tiers(s0, s0['proba'].to_numpy())
s1['tier'] = tiers(s1, s1['proba'].to_numpy())
assert (s0['tier'].to_numpy() == hold_tiers['tier'].to_numpy()).all(), 'S0 등급이 03 노트북과 다름'

# 2차 체크포인트: 판정 불명 학생은 취소일이 없으므로 체크포인트 재학으로 간주하고 미제출 여부만 계산
cp_def = pd.DataFrame(cp_rule['checkpoints'])
post25 = assessments[assessments['assessment_type'].isin(['TMA', 'CMA']) & (assessments['date'] > 25)]
due = post25.merge(cp_def[['code_module', 'code_presentation', 'checkpoint_day']], on=['code_module', 'code_presentation'])
due = due[due['date'] == due['checkpoint_day']]
done = student_assessment.merge(due[['id_assessment', 'code_module', 'code_presentation', 'checkpoint_day']], on='id_assessment')
done = done[(done['is_banked'] == 1) | (done['date_submitted'] <= done['checkpoint_day'])].groupby(KEYS).size().rename('n_done').reset_index()
amb_cp = amb_j[KEYS].merge(cp_def[['code_module', 'code_presentation', 'same_day_n']], on=['code_module', 'code_presentation']).merge(done, on=KEYS, how='left')
amb_cp['missed_cp'] = amb_cp['n_done'].fillna(0) < amb_cp['same_day_n']
hold_cp = cp_students[cp_students['세트'] == 'holdout'][KEYS + ['enrolled_at_cp', 'missed_cp']]
cp_all = pd.concat([hold_cp, amb_cp[KEYS + ['missed_cp']].assign(enrolled_at_cp=True)], ignore_index=True)
s0 = s0.merge(hold_cp, on=KEYS, how='left', validate='one_to_one')
s1 = s1.merge(cp_all, on=KEYS, how='left', validate='one_to_one')


def metrics(frame):
    y = frame[TARGET].to_numpy() == 1
    p = frame['proba'].to_numpy()
    t = frame['tier'].to_numpy()
    promote = (t == 3) & frame['enrolled_at_cp'].to_numpy(dtype=bool) & frame['missed_cp'].to_numpy(dtype=bool)
    staff = (t <= 2) | promote
    out = {'학생': len(frame), '이탈': int(y.sum()), '이탈률': y.mean(), 'PR-AUC': average_precision_score(y, p), 'PR-AUC/이탈률': average_precision_score(y, p) / y.mean(),
           'ROC-AUC': roc_auc_score(y, p), 'Brier': brier_score_loss(y, p), '평균 예측 − 실제': p.mean() - y.mean(),
           '관리자 명단(25일)': (t <= 2).mean(), '25일 명단 포착률': (y & (t <= 2)).sum() / y.sum(),
           '1차+2차 명단': staff.mean(), '1차+2차 포착률': (y & staff).sum() / y.sum(), '1차+2차 적중률': y[staff].mean()}
    for k in [1, 2, 3, 4]:
        out[f'{TIER_NAMES[k]} 이탈률'] = y[t == k].mean()
    return out


sens = pd.DataFrame({'S0 제외(주 분석)': metrics(s0), 'S1 87명 포함(상한)': metrics(s1)}).T
sens.loc['차이(S1 − S0)'] = sens.iloc[1] - sens.iloc[0]
fmt = {c: '{:.4f}' for c in ['PR-AUC', 'ROC-AUC', 'Brier']} | {'PR-AUC/이탈률': '{:.2f}', '학생': '{:,.0f}', '이탈': '{:,.0f}'}
fmt |= {c: '{:+.1%}' if c == '평균 예측 − 실제' else '{:.1%}' for c in sens.columns if c not in fmt}
show(sens, fmt)

amb_tier = pd.Series(s1.loc[s1['판정불명'], 'tier']).map(TIER_NAMES).value_counts().reindex(TIER_NAMES.values()).fillna(0).astype(int)
amb_promote = int(((s1['판정불명']) & (s1['tier'] == 3) & s1['missed_cp'].astype(bool)).sum())
print('판정 불명 87명의 등급 분포:', amb_tier.to_dict(), '| 2차 승격 대상:', amb_promote)

판정 불명 87명 보정 확률: 평균 37.5%, 중앙값 38.1% (holdout 이탈자 평균 24.9%, 비이탈자 16.4%)


,학생,이탈,이탈률,PR-AUC,PR-AUC/이탈률,ROC-AUC,Brier,평균 예측 − 실제,관리자 명단(25일),25일 명단 포착률,1차+2차 명단,1차+2차 포착률,1차+2차 적중률,1 고위험 이탈률,2 주의 이탈률,3 관찰 이탈률,4 일반 이탈률
S0 제외(주 분석),"9,150","1,720",18.8%,0.3572,1.90,0.6794,0.1421,-0.8%,34.2%,56.1%,39.6%,63.8%,30.2%,47.0%,26.9%,17.7%,10.1%
S1 87명 포함(상한),"9,237","1,807",19.6%,0.3822,1.95,0.6889,0.1446,-1.4%,34.8%,57.8%,40.2%,65.3%,31.8%,49.9%,28.2%,17.9%,10.1%
차이(S1 − S0),87,87,0.8%,0.0250,0.05,0.0095,0.0025,-0.6%,0.5%,1.7%,0.5%,1.5%,1.6%,3.0%,1.3%,0.2%,0.1%


판정 불명 87명의 등급 분포: {'1 고위험': 36, '2 주의': 44, '3 관찰': 4, '4 일반': 3} | 2차 승격 대상: 3


**해석 — holdout 결과는 견고하다**

| 지표 | S0 제외(주 분석) | S1 87명 포함(상한) | 차이 |
|---|---:|---:|---:|
| PR-AUC (이탈률 대비) | 0.357 (1.90배) | 0.382 (1.95배) | +0.025 |
| 평균 예측 − 실제 | −0.8%p | −1.4%p | −0.6%p |
| 25일 명단 포착률 | 56.1% | 57.8% | +1.7%p |
| 1차+2차 포착률 | 63.8% | 65.3% | +1.5%p |
| 등급별 이탈률 (고위험/주의/관찰/일반) | 47.0 / 26.9 / 17.7 / 10.1% | 49.9 / 28.2 / 17.9 / 10.1% | ≤ +3.0%p |

- **87명을 모두 이탈자로 넣어도 결론이 바뀌지 않는다.** 성능·포착률은 소폭 **올라가고**, 등급별 이탈률은 ±3%p 안에서 유지된다.
- 모델은 이 87명에게 평균 **37.5%**의 이탈 확률을 줬다(holdout 이탈자 평균 24.9%). **87명 중 80명(92%)이 고위험·주의 등급**이다.
  이들이 실제로 25일 재학 중이었다면 시스템이 거의 다 잡았을 것이다.
- 즉 이 87명이 실제 "25일 이후 이탈"이라면 **주 분석(S0)의 holdout 수치는 보수적인 쪽**이고, 25일 전 이탈이라면 제외가 맞다. **어느 경우든 판정 불명 학생을 제외해 성능을 부풀린 것은 아니다.**

## 5. 개발 세트 판정 불명(6명)

개발 학기에는 6명뿐이다(학기당 0–2명). 모델 학습·등급 기준 결정에 미치는 영향은 무시할 수 있는 규모라 별도 재학습은 하지 않는다.

In [5]:
dev_amb = feat_amb[feat_amb['code_presentation'] != '2014J']
print(f'개발 학기 판정 불명 {len(dev_amb)}명 (개발 모델 대상 {len(dev):,}명의 {len(dev_amb) / len(dev):.3%}, 개발 이탈자의 {len(dev_amb) / y_dev.sum():.2%})')
print(dev_amb.groupby(['code_module', 'code_presentation']).size().to_dict())

개발 학기 판정 불명 6명 (개발 모델 대상 18,504명의 0.032%, 개발 이탈자의 0.17%)
{('BBB', '2014B'): 1, ('DDD', '2013B'): 1, ('DDD', '2014B'): 1, ('FFF', '2013J'): 1, ('FFF', '2014B'): 1, ('GGG', '2013J'): 1}


## 6. 결과 저장

In [6]:
sens.to_csv(MODEL_DIR / 'sensitivity_ambiguous_withdrawn_n25.csv', encoding='utf-8-sig')
feat_amb.merge(s1.loc[s1['판정불명'], KEYS + ['proba', 'tier', 'missed_cp']], on=KEYS, how='left').to_csv(MODEL_DIR / 'ambiguous_withdrawn_features_n25.csv', index=False, encoding='utf-8-sig')
print('saved')

saved


## 7. 결론 (열린 이슈 해소)

1. **제외 결정은 타당하다**: 판정 불명 93명의 25일 행동(활동일 중앙값 1일, 미제출 78.8%)은 25일 전 이탈자와 비슷하다. 25일 재학생으로 보기 어렵다.
2. **holdout 결과는 견고하다**: 2014J의 87명을 "25일 재학 후 이탈"로 모두 포함하는 상한 시나리오에서도 PR-AUC는 0.357 → 0.382로 오르고, 포착률은 +1.5–1.7%p, 등급별 이탈률은 ±3%p 안에서 유지된다.
3. **제외로 성능을 부풀리지 않았다**: 포함하면 수치가 오히려 좋아지므로, 주 분석 수치는 보수적인 쪽이다. 모델은 이 학생들 대부분(92%)을 고위험·주의로 표시했다.
4. **재현성**: 제외 세그먼트용 25일 피처 재계산 함수가 통합 정본 27,661건의 피처 12개와 불일치 0건으로 일치함을 확인했다.
5. 개발 학기의 판정 불명 6명(개발 이탈자의 0.17%)은 모델·기준 결정에 영향이 없는 규모다.

→ HANDOFF 열린 이슈 "판정 불명 Withdrawn 93건 민감도 분석"을 **해소**로 처리한다.